# Часть 2 — LLM в LangGraph подключается к MCP

В части 1 инструменты вызывали мы. Теперь их будет вызывать **модель**: сама решит, какой инструмент
нужен, с какими аргументами, и соберёт ответ.

Шаги:

1. модель;
2. MCP-клиент для LangChain — превращает инструменты сервера в обычные LangChain-инструменты;
3. граф LangGraph — тот же «агент ⇄ инструменты», что в занятии 1;
4. спрашиваем.

## Шаг 1. Модель

In [1]:
import os
import sys

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("MODEL", "openai/gpt-4o-mini"),
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

## Шаг 2. MCP-клиент → инструменты LangChain

`MultiServerMCPClient` (библиотека `langchain-mcp-adapters`) делает за нас всё из части 1: запускает
сервер, проходит рукопожатие, берёт `list_tools()` — и отдаёт **обычные LangChain-инструменты**.
Серверов можно указать сколько угодно — каждый под своим именем.

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "bank": {"transport": "stdio", "command": sys.executable, "args": ["db_server.py"]}
})

tools = await client.get_tools()
for tool in tools:
    print(f"{tool.name:<15} {tool.description}")

list_tables     List the tables of the bank database with a short description of each.
describe_table  Show the columns of one table: name, type and what the column means.
run_sql         Run a read-only SQL query (PostgreSQL) and return up to 50 rows.


In [3]:
import subprocess
import time

from langchain_mcp_adapters.client import MultiServerMCPClient

# 1. сервер: запускаем отдельным процессом, он слушает порт 8011
server = subprocess.Popen([sys.executable, "db_server.py", "--http"])
time.sleep(3)  # даём ему время запуститься

# 2. клиент: подключается к уже работающему серверу по адресу
client = MultiServerMCPClient(
    {
        "bank": {"transport": "streamable_http", "url": "http://localhost:8011/mcp"},
    }
)

tools = await client.get_tools()
for tool in tools:
    print(f"{tool.name:<15} {tool.description}")

list_tables     List the tables of the bank database with a short description of each.
describe_table  Show the columns of one table: name, type and what the column means.
run_sql         Run a read-only SQL query (PostgreSQL) and return up to 50 rows.


In [4]:
server.terminate()

Для HTTP-сервера конфиг был бы таким — остальной код не меняется:

```python
"bank": {"transport": "streamable_http", "url": "http://localhost:8011/mcp"}
```

## Шаг 3. Граф LangGraph

Ровно граф из занятия 1 (`06_langgraph`): узел `agent` зовёт модель, узел `tools` выполняет
инструменты, ребро `tools_condition` решает — нужны ещё инструменты или ответ готов.
Единственное отличие: инструменты пришли с MCP-сервера, а не из нашего `@tool`.

In [9]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

SYSTEM = ("Ты аналитик банка. Отвечай по-русски, кратко и с цифрами. Суммы — в чешских кронах (CZK). "
          "Структуру базы узнавай через list_tables и describe_table, данные — через run_sql.")

llm_with_tools = llm.bind_tools(tools)          # the model now knows the MCP tools


async def agent(state: MessagesState) -> dict:
    response = await llm_with_tools.ainvoke([("system", SYSTEM), *state["messages"]])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))       # runs the tools = calls the MCP server
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
graph = builder.compile(checkpointer=InMemorySaver())   # memory: remembers the conversation

## Шаг 4. Спрашиваем

Маленький помощник `ask()` печатает каждый шаг: что модель решила вызвать (🤖 →) и что ответил
сервер (🔧).

In [10]:
CONFIG = {"configurable": {"thread_id": "demo"}}   # one conversation


async def ask(question: str) -> None:
    async for step in graph.astream({"messages": [("user", question)]}, CONFIG, stream_mode="updates"):
        for node, update in step.items():
            for message in update["messages"]:
                if message.type == "ai" and message.tool_calls:
                    for call in message.tool_calls:
                        print(f"🤖 → {call['name']}({call['args']})")
                elif message.type == "tool":
                    print(f"🔧 {message.name}: {message.text[:200]}")
                else:
                    print(f"\n💬 {message.text}")

In [11]:
await ask("Сколько в банке клиентов?")

🤖 → list_tables({})
🔧 list_tables: table_name | description
accounts | Bank accounts. Balance is NOT stored here — take the balance of the latest transaction.
cards | Payment cards issued to an account disposition.
clients | Bank clien
🤖 → describe_table({'table': 'clients'})
🔧 describe_table: column_name | data_type | meaning
client_id | integer | Client id.
gender | character | M or F.
birth_date | date | Date of birth.
district_id | integer | Where the client lives → districts.
🤖 → run_sql({'sql': 'SELECT COUNT(*) AS client_count FROM clients;'})
🔧 run_sql: client_count
5369

💬 В банке **5 369 клиентов**.


Модель не знала устройство базы — и сама вызвала `list_tables`, а потом написала SQL.
Что произошло по шагам:

```
вопрос → agent (LLM): «нужен list_tables»  → tools → MCP-сервер → Postgres
       → agent (LLM): «нужен run_sql(...)»  → tools → MCP-сервер → Postgres
       → agent (LLM): готовый ответ
```

In [12]:
await ask("Сколько кредитов в каждом статусе?")

🤖 → describe_table({'table': 'loans'})
🔧 describe_table: column_name | data_type | meaning
loan_id | integer | Loan id.
account_id | integer | → accounts.
granted_on | date | Date the loan was granted.
amount | numeric | Loan amount, CZK.
duration_months | 
🤖 → run_sql({'sql': 'SELECT status, COUNT(*) AS loan_count FROM loans GROUP BY status ORDER BY status;'})
🔧 run_sql: status | loan_count
finished_default | 31
finished_ok | 203
running_in_debt | 45
running_ok | 403

💬 Количество кредитов по статусам:

- `finished_ok` — 203
- `finished_default` — 31
- `running_ok` — 403
- `running_in_debt` — 45


Тот же разговор (`thread_id`) — агент помнит, о каких кредитах речь:

In [13]:
await ask("А какая доля из них проблемных — не возвращены или просрочены?")

🤖 → run_sql({'sql': "SELECT COUNT(*) AS total_loans, COUNT(*) FILTER (WHERE status IN ('finished_default', 'running_in_debt')) AS problematic_loans, ROUND(100.0 * COUNT(*) FILTER (WHERE status IN ('finished_default', 'running_in_debt')) / COUNT(*), 1) AS problematic_share_pct FROM loans;"})
🔧 run_sql: total_loans | problematic_loans | problematic_share_pct
682 | 76 | 11.1

💬 Проблемных кредитов **76 из 682**, или **11,1%**. Сюда входят невозвращённые (`finished_default`) и просроченные (`running_in_debt`).


## Итоги

- `MultiServerMCPClient` → `get_tools()` — и инструменты MCP-сервера стали обычными LangChain-инструментами;
- граф LangGraph тот же, что в занятии 1; модель сама выбирает инструменты и пишет SQL;
- **проверяйте**: агент пишет SQL сам и может ошибиться — поэтому мы печатаем каждый его шаг.

Дальше — часть 3: закрываем сервер паролем (токеном), чтобы им пользовались только свои.